# EnergyPulse: Exploratory Data Analysis

Goal: understand the raw UCI data well enough to choose cleaning rules, then check the cleaned hourly data.
Run from the project root after `pip install -e .`.

In [34]:
import pandas as pd
import plotly.express as px

from energypulse.pipeline import load_raw, clean_minutes, to_hourly, fill_gaps
import sqlite3, pandas as pd
con = sqlite3.connect("../data/processed/energypulse.db")
pd.read_sql("select * from hourly_readings", con, parse_dates=["timestamp"]).tail()
RAW_PATH = "../data/raw/household_power_consumption.txt"
pd.options.display.float_format = "{:,.3f}".format

## 1. Load the raw data

In [20]:
raw = load_raw(RAW_PATH)
print(raw.shape)
print("Range:", raw.index.min(), "->", raw.index.max())
raw.head()

(2075259, 7)
Range: 2006-12-16 17:24:00 -> 2010-11-26 21:02:00


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
timestamp,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.000
2006-12-16 17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.000
2006-12-16 17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.000
2006-12-16 17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.000
2006-12-16 17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.000


In [21]:
raw.dtypes

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

## 2. Missing values (minute level)

In [22]:
# Share of missing values per column, and how many timestamps failed to parse
print(raw.isna().mean().round(4))
print("Unparseable timestamps:", raw.index.isna().sum())

Global_active_power     0.013
Global_reactive_power   0.013
Voltage                 0.013
Global_intensity        0.013
Sub_metering_1          0.013
Sub_metering_2          0.013
Sub_metering_3          0.013
dtype: float64
Unparseable timestamps: 0


In [23]:
# Which years/months lose the most data?
missing_by_month = raw["Global_active_power"].isna().groupby(raw.index.to_period("M")).mean()
px.bar(missing_by_month.rename("share_missing").reset_index().astype({"timestamp": str}),
       x="timestamp", y="share_missing", title="Share of missing minutes per month")

### Gap lengths
Are gaps short blips or long outages? This decides `MAX_INTERP_GAP_HOURS` in `pipeline.py`.

In [ ]:
minutes = clean_minutes(raw)
series = minutes["Global_active_power"].asfreq("min")  # make absent minutes explicit, frequency
missing = series.isna()  ## where missing(Na) it is True, otherwise False
run_id = (~missing).cumsum() ##  unique id for not missing values is a sum of them, when missing values come they get the same id until the next not missing
gap_minutes = missing.groupby(run_id).sum() ## group by the unique id and suming the missing(true(1)) values which gives the gap of minutes
gap_minutes = gap_minutes[gap_minutes > 0] 

print(gap_minutes.describe()) ## statistics about gap minutes
print("Gaps longer than 3 hours:", (gap_minutes > 180).sum(), "of", len(gap_minutes))
gap_minutes.sort_values(ascending=False).head(10) ## the 10 longest gap in descending order

count      71.000
mean      365.901
std     1,251.468
min         1.000
25%         1.000
50%         1.000
75%         3.000
max     7,226.000
Name: Global_active_power, dtype: float64
Gaps longer than 3 hours: 7 of 71


Global_active_power
1916303    7226
1964211    5237
190488     3723
1305243    3305
1608618    3129
1701305    2027
1390044     891
300030       83
1039605      70
300039       47
Name: Global_active_power, dtype: int64

In [25]:
gap_df = gap_minutes[gap_minutes <= 600].rename("gap_minutes").reset_index(drop=True).to_frame()
px.histogram(gap_df, x="gap_minutes", nbins=60,
             title="Gap length in minutes (gaps up to 10 hours)")

## 3. Sanity-check value ranges

In [26]:
# Do the VALID_RANGES in pipeline.py match reality? Compare raw vs cleaned.
numeric_raw = raw.apply(pd.to_numeric, errors="coerce")
pd.concat({"raw": numeric_raw.describe().T[["min", "max"]],
           "cleaned": minutes.describe().T[["min", "max"]]}, axis=1)

raw         cleaned        
                          min     max     min     max
Global_active_power     0.076  11.122   0.076  11.122
Global_reactive_power   0.000   1.390   0.000   1.390
Voltage               223.200 254.150 223.200 254.150
Global_intensity        0.200  48.400   0.200  48.400
Sub_metering_1          0.000  88.000   0.000  88.000
Sub_metering_2          0.000  80.000   0.000  80.000
Sub_metering_3          0.000  31.000   0.000  31.000

In [27]:
# How many values did cleaning remove, per column?
(numeric_raw.notna().sum() - minutes.notna().sum())

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

## 4. Hourly data after cleaning

In [28]:
hourly_raw = to_hourly(minutes)
hourly = fill_gaps(hourly_raw)
print("Hours:", len(hourly))
print("Imputed:", hourly["is_imputed"].sum(), "| Still missing:", hourly["is_long_gap"].sum())
hourly.describe().T

Hours: 34589
Imputed: 8 | Still missing: 430


,count,mean,std,min,25%,50%,75%,max
energy_kwh,"34,159.000",1.092,0.898,0.124,0.342,0.803,1.579,6.561
reactive_kvar,"34,159.000",0.124,0.067,0.013,0.078,0.107,0.149,0.774
voltage,"34,159.000",240.840,3.000,225.834,239.181,240.979,242.695,251.902
intensity,"34,159.000",4.628,3.757,0.503,1.513,3.423,6.607,28.383
sub1_wh,"34,159.000",67.318,212.286,0.000,0.000,0.000,0.000,"2,902.000"
sub2_wh,"34,159.000",77.893,251.570,0.000,0.000,20.000,39.000,"2,786.000"
sub3_wh,"34,159.000",387.468,440.990,0.000,39.000,99.000,853.000,"1,293.000"
minutes_observed,"34,589.000",59.247,6.636,0.000,60.000,60.000,60.000,60.000


## 5. Look at the data: a few days and weeks

In [29]:
week = hourly.loc["2007-03-05":"2007-03-11"]
px.line(week.reset_index(), x="timestamp", y="energy_kwh", title="One week of hourly consumption (kWh)")

In [30]:
px.line(hourly["energy_kwh"].resample("D").sum().reset_index(),
        x="timestamp", y="energy_kwh", title="Daily consumption (kWh)")

### Daily and weekly patterns (these motivate the forecasting features)

In [31]:
by_hour = hourly.groupby(hourly.index.hour)["energy_kwh"].mean()
px.line(by_hour.reset_index(), x="timestamp", y="energy_kwh",
        labels={"timestamp": "hour of day"}, title="Average consumption by hour of day")

In [33]:
# Daily totals: keep only complete days
# 1. Napi szummák és óraszámok kigyűjtése
d = hourly["energy_kwh"].resample("D").agg(["sum", "count"])

# 2. Csak a teljes (24 órás) napok megtartása, a többi törlése (dropna)
daily_complete = d[d["count"] == 24][["sum"]].rename(columns={"sum": "energy_kwh"})

# 3. Csoportosítás a hét napjai szerint és az átlag (mean) kiszámítása
weekday_avg = daily_complete.groupby(daily_complete.index.day_name()).mean()

# 4. Sorbarendezés hétfőtől vasárnapig
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_avg = weekday_avg.reindex(order)

# 5. Plotly grafikon kirajzolása (most már létezik az 'energy_kwh' oszlop)
fig = px.bar(
    weekday_avg.reset_index(), 
    x="timestamp",             # A .reset_index() után az indexből (napok nevei) ez az oszlop lesz
    y="energy_kwh", 
    labels={"timestamp": "weekday", "energy_kwh": "Average Energy (kWh)"}, 
    title="Average consumption by weekday"
)
fig.show()


In [ ]:
monthly = hourly["energy_kwh"].resample("MS").mean() * 24
px.line(monthly.reset_index(), x="timestamp", y="energy_kwh", title="Monthly consumption (seasonality)")

## 6. Findings to carry into the pipeline
Missing data: 1.3% of minutes are missing, and the share is identical across all columns. That means the meter went offline for a while, not that one sensor failed.
Gap shape: there are 71 gaps, and they fall into two groups. 64 are short (at most 83 minutes, and 38 are exactly one minute). The other 7 are long, from about 15 hours to 5 days, and nothing falls in between. The 7 long gaps account for almost all of the missing time, about 426 hours.
Where the long gaps are: I matched them by size against your monthly missing-data chart. They fall in Apr 2007, Jun 2009, Aug 2009, Jan 2010, Mar 2010, Aug 2010 (the 5-day one) and Sep 2010.
Your 3-hour limit is fine: because of that empty middle, any limit between about 2 and 14 hours gives identical results.
Only 8 hours were imputed. An hour counts as valid when it has at least 45 of 60 minutes. So short gaps are absorbed by averaging within the hour, and interpolation rarely kicks in. The 430 hours still flagged is_long_gap (1.2% of all hours) are essentially the 7 long outages. The pipeline is working as designed.
Cleaning removed 0 values. The UCI data is physically clean apart from the ? markers, so VALID_RANGES is just a safety net. The real maximum for Sub_metering_1 is 88, under my cap of 100.